<a href="https://colab.research.google.com/github/azcsprof/UCLA-XL161/blob/Final-Project-Starter-Code/XL161_Final_Project_Option_3_Starter_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# COM SCI XL161 FINAL PROJECT — OPTION 3
# Learning-Based Game Agent
# ============================================================
#
# Core Idea:
#
# This project models a simple adversarial AI system.
#
# The agent plays a turn-based game against an opponent.
#
# The system must:
#
#   • search future moves
#   • evaluate board states
#   • reason about threats
#   • handle uncertainty
#   • learn from previous games
#
# The important question is NOT:
#
#     "Did the AI win?"
#
# The important question is:
#
#     "How did the system decide what move to make?"
#
# ============================================================
#
# MODULE CONNECTIONS
#
# Module 1:
#   Agents, environments, actions, goals
#
# Module 2:
#   Search through possible future states
#
# Module 3:
#   Heuristic evaluation
#
# Module 4:
#   Constraint-based move validation
#
# Module 5:
#   Adversarial reasoning / minimax
#
# Module 6:
#   Rule-based reasoning
#
# Module 7:
#   Structured board representation
#
# Module 8:
#   Uncertainty and probabilistic prediction
#
# Module 9:
#   Learning from previous outcomes
#
# ============================================================


import random


# ============================================================
# 1. GAME BOARD
# ============================================================
#
# The board is represented as a list.
#
# Each position may contain:
#
#   "X" = AI agent
#   "O" = opponent
#   " " = empty
#
# This representation defines the environment the agent reasons about.
#
# ============================================================

board = [" ", " ", " ",
         " ", " ", " ",
         " ", " ", " "]


# ============================================================
# 2. WINNING COMBINATIONS
# ============================================================
#
# These combinations define success conditions.
#
# A game state is successful if:
#
#   one player controls all positions in a winning line.
#
# ============================================================

winning_lines = [
    [0, 1, 2],
    [3, 4, 5],
    [6, 7, 8],

    [0, 3, 6],
    [1, 4, 7],
    [2, 5, 8],

    [0, 4, 8],
    [2, 4, 6]
]


# ============================================================
# 3. LEARNED MOVE PREFERENCES
# ============================================================
#
# Module 9 connection:
#
# The system can learn preferred board positions.
#
# If certain moves repeatedly lead to success,
# their preference score increases.
#
# This creates adaptation over time.
#
# ============================================================

learned_move_scores = {
    0: 0,
    1: 0,
    2: 0,
    3: 0,
    4: 0,
    5: 0,
    6: 0,
    7: 0,
    8: 0
}


# ============================================================
# 4. DISPLAY BOARD
# ============================================================
#
# This function prints the board for humans.
#
# The AI itself reasons using the internal representation,
# not the printed output.
#
# ============================================================

def print_board(current_board):

    print("\n")

    print(current_board[0], "|", current_board[1], "|", current_board[2])
    print("--+---+--")

    print(current_board[3], "|", current_board[4], "|", current_board[5])
    print("--+---+--")

    print(current_board[6], "|", current_board[7], "|", current_board[8])

    print("\n")


# ============================================================
# 5. CHECK FOR VALID MOVES
# ============================================================
#
# Module 4 connection:
#
# Constraints eliminate illegal actions.
#
# A move is invalid if:
#
#   the board position is already occupied.
#
# ============================================================

def get_valid_moves(current_board):

    valid_moves = []

    for i in range(len(current_board)):

        if current_board[i] == " ":
            valid_moves.append(i)

    return valid_moves


# ============================================================
# 6. CHECK WINNER
# ============================================================
#
# This function determines whether a player has achieved
# a winning state.
#
# ============================================================

def check_winner(current_board, player):

    for line in winning_lines:

        if (
            current_board[line[0]] == player and
            current_board[line[1]] == player and
            current_board[line[2]] == player
        ):
            return True

    return False


# ============================================================
# 7. TERMINAL STATE CHECK
# ============================================================
#
# A terminal state means:
#
#   • someone won
#   • or the board is full
#
# No additional moves are possible.
#
# ============================================================

def game_over(current_board):

    if check_winner(current_board, "X"):
        return True

    if check_winner(current_board, "O"):
        return True

    if " " not in current_board:
        return True

    return False


# ============================================================
# 8. RULE-BASED REASONING
# ============================================================
#
# Module 6 connection:
#
# The agent applies symbolic IF/THEN rules.
#
# Example:
#
# IF opponent can win next turn
# THEN block opponent.
#
# This reasoning is NOT learned.
# These are explicitly encoded rules.
#
# ============================================================

def find_blocking_move(current_board):

    for move in get_valid_moves(current_board):

        test_board = current_board.copy()

        test_board[move] = "O"

        if check_winner(test_board, "O"):
            return move

    return None


# ============================================================
# 9. STRUCTURED BOARD REPRESENTATION
# ============================================================
#
# Module 7 connection:
#
# Instead of treating the board as isolated positions,
# the system reasons about relationships:
#
#     controls(X, center)
#     threatens(O, top_row)
#
# This creates structured relational reasoning.
#
# ============================================================

def describe_board_structure(current_board):

    print("\n=== STRUCTURED BOARD KNOWLEDGE ===")

    positions = [
        "top_left",
        "top_middle",
        "top_right",
        "middle_left",
        "center",
        "middle_right",
        "bottom_left",
        "bottom_middle",
        "bottom_right"
    ]

    for i in range(len(current_board)):

        if current_board[i] != " ":

            print(
                f"controls({current_board[i]}, {positions[i]})"
            )


# ============================================================
# 10. HEURISTIC EVALUATION
# ============================================================
#
# Module 3 connection:
#
# The system estimates how favorable a board state is.
#
# This heuristic guides decision-making when full search
# becomes expensive.
#
# Higher scores = better for AI
# Lower scores = better for opponent
#
# ============================================================

def evaluate_board(current_board):

    score = 0

    # Center control is strategically valuable.
    if current_board[4] == "X":
        score += 3

    if current_board[4] == "O":
        score -= 3

    # Reward AI for controlling corners.
    corners = [0, 2, 6, 8]

    for corner in corners:

        if current_board[corner] == "X":
            score += 1

        if current_board[corner] == "O":
            score -= 1

    return score


# ============================================================
# 11. PROBABILISTIC OPPONENT MODEL
# ============================================================
#
# Module 8 connection:
#
# Real opponents are not perfectly predictable.
#
# Instead of assuming certainty,
# the AI estimates probabilities of dangerous moves.
#
# This creates reasoning under uncertainty.
#
# ============================================================

def estimate_opponent_threat(move):

    # Random uncertainty model.
    #
    # In more advanced systems,
    # probabilities would come from data.
    #
    threat_probability = random.uniform(0, 1)

    return threat_probability


# ============================================================
# 12. MINIMAX-STYLE DECISION PROCESS
# ============================================================
#
# Module 5 connection:
#
# The system searches future possibilities
# while assuming the opponent may respond strategically.
#
# This is a simplified minimax-style decision process.
#
# ============================================================

def choose_best_move(current_board):

    print("\n=== AI DECISION PROCESS ===")

    best_move = None
    best_score = -999

    # First apply symbolic blocking rules.
    blocking_move = find_blocking_move(current_board)

    if blocking_move is not None:

        print("Rule triggered:")
        print("  Opponent threat detected.")
        print(f"  Blocking move selected: {blocking_move}")

        return blocking_move

    # Otherwise evaluate all possible moves.
    for move in get_valid_moves(current_board):

        simulated_board = current_board.copy()

        simulated_board[move] = "X"

        heuristic_score = evaluate_board(simulated_board)

        learned_bonus = learned_move_scores[move]

        threat_probability = estimate_opponent_threat(move)

        # Higher threat probability reduces score.
        uncertainty_penalty = threat_probability * 2

        final_score = (
            heuristic_score
            + learned_bonus
            - uncertainty_penalty
        )

        print(f"\nEvaluating move: {move}")
        print(f"  Heuristic score: {heuristic_score}")
        print(f"  Learned bonus: {learned_bonus}")
        print(f"  Threat probability: {threat_probability:.2f}")
        print(f"  Final score: {final_score:.2f}")

        if final_score > best_score:

            best_score = final_score
            best_move = move

    return best_move


# ============================================================
# 13. LEARNING UPDATE
# ============================================================
#
# Module 9 connection:
#
# After a game ends,
# the system updates move preferences.
#
# Moves associated with success become more preferred.
#
# ============================================================

def update_learning(ai_moves, ai_won):

    print("\n=== LEARNING UPDATE ===")

    for move in ai_moves:

        if ai_won:

            learned_move_scores[move] += 1

            print(f"Move {move} rewarded.")

        else:

            learned_move_scores[move] -= 1

            print(f"Move {move} penalized.")

    print("\nUpdated move preferences:")

    for move, score in learned_move_scores.items():

        print(f"  Position {move}: {score}")


# ============================================================
# 14. OPPONENT MOVE
# ============================================================
#
# The opponent selects a random valid move.
#
# This creates adversarial interaction.
#
# ============================================================

def opponent_move(current_board):

    possible_moves = get_valid_moves(current_board)

    return random.choice(possible_moves)


# ============================================================
# 15. FULL GAME LOOP
# ============================================================
#
# This function runs the full game:
#
#   • search
#   • heuristics
#   • rules
#   • uncertainty
#   • learning
#
# ============================================================

def play_game():

    current_board = [" ", " ", " ",
                     " ", " ", " ",
                     " ", " ", " "]

    ai_moves = []

    print("\n================================================")
    print("LEARNING-BASED GAME AGENT")
    print("================================================")

    while not game_over(current_board):

        print_board(current_board)

        describe_board_structure(current_board)

        # AI TURN
        move = choose_best_move(current_board)

        current_board[move] = "X"

        ai_moves.append(move)

        print(f"\nAI selects move: {move}")

        if check_winner(current_board, "X"):

            print_board(current_board)

            print("AI wins.")

            update_learning(ai_moves, ai_won=True)

            return

        if game_over(current_board):
            break

        # OPPONENT TURN
        opponent_choice = opponent_move(current_board)

        current_board[opponent_choice] = "O"

        print(f"\nOpponent selects move: {opponent_choice}")

        if check_winner(current_board, "O"):

            print_board(current_board)

            print("Opponent wins.")

            update_learning(ai_moves, ai_won=False)

            return

    print_board(current_board)

    print("Game ended in a draw.")

    update_learning(ai_moves, ai_won=False)


# ============================================================
# 16. RUN MULTIPLE GAMES
# ============================================================
#
# Run several games and observe:
#
#   • changing move preferences
#   • strategic adaptation
#   • interaction between rules and learning
#
# ============================================================

play_game()

print("\n\nSECOND GAME AFTER LEARNING:\n")

play_game()

# STUDENT EXTENSIONS
#
# 1. Improve heuristic evaluation.
# 2. Add deeper minimax search.
# 3. Add better opponent prediction.
# 4. Add stronger symbolic rules.
# 5. Change uncertainty model.
# 6. Add probabilistic bluffing behavior.
# 7. Break the system intentionally:
#       bad heuristics
#       incorrect rules
#       poor learning updates
# 8. Improve the system afterward.